# MARIS Data Guide

> A reference for data providers and data users

## About MARIS

The [IAEA Marine Radioactivity Information System (MARIS)](https://maris.iaea.org) provides open access to radioactivity measurements in marine environments worldwide. Developed and maintained by the [IAEA Marine Environmental Laboratories](https://www.iaea.org/about/organizational-structure/department-of-nuclear-sciences-and-applications/division-of-iaea-environment-laboratories) in Monaco, MARIS integrates measurements from regional monitoring programmes, event-driven datasets, and historical scientific literature into a single, freely accessible database.

## Sample types

All measurements are organized into four sample type groups:

| Sample type | What it covers | Primary unit |
|---|---|---|
| **SEAWATER** | Dissolved and filtered water samples (seawater and brackish water) | Bq m⁻³ |
| **BIOTA** | Marine organisms — fish, shellfish, seaweed, and other biota | Bq kg⁻¹ (wet or dry weight) |
| **SEDIMENT** | Bottom sediments, including core profiles | Bq kg⁻¹ or Bq m⁻² |
| **SUSPENDED** | Suspended particulate matter in the water column | Bq kg⁻¹ |

## Measurement fields

Each record contains:

- **Location** — latitude, longitude, station name, named area
- **Time** — sampling date (encoded as seconds since 1970-01-01 UTC in NetCDF4)
- **Depth** — sampling depth and total water column depth
- **Nuclide** — identity via a MARIS standard ID linked to a lookup table
- **Activity** — measured value or minimum detectable activity (MDA)
- **Uncertainty** — 1σ absolute uncertainty in the same unit as the activity
- **Detection limit flag** — `=` (measured), `<` (below detection limit), `ND` (not detected), or `DE` (derived/aggregated)
- **Unit** — MARIS standard unit ID from a lookup table
- Sample-type-specific fields: species and body part (BIOTA); sediment type and core interval (SEDIMENT); filter status (SEAWATER)
- Method fields: sampling method, preparation method, counting method

→ See [Field Definitions](field-definition.ipynb) for the complete field reference.  
→ See [Sample ID Coverage](sample-id-coverage.ipynb) for the status of `SMP_ID` / `SMP_ID_PROVIDER` capture across all handlers.

## Nomenclature and lookup tables

Enumerated fields (nuclide identity, species, units, sediment type, body part, etc.) are stored as integer IDs referencing standardised lookup tables maintained by MARIS. This design ensures consistency across datasets from different providers.

| Field | Lookup table |
|---|---|
| Nuclide identity | `dbo_nuclide.xlsx` |
| Unit | `dbo_unit.xlsx` |
| Species | `dbo_species.xlsx` |
| Biological group | derived from `dbo_species.xlsx` |
| Body part | `dbo_bodypar.xlsx` |
| Sediment type | `dbo_sedtype.xlsx` |
| Detection limit | `dbo_detectlimit.xlsx` |
| Laboratory | `dbo_lab.xlsx` |
| Sampling method | `dbo_sampmet.xlsx` |
| Preparation method | `dbo_prepmet.xlsx` |
| Counting method | `dbo_counmet.xlsx` |
| Named area | `dbo_area.xlsx` |

## Data formats

MARIS data is available in two formats:

**NetCDF4** (primary format)  
Each file is self-contained: measurements, variable metadata, lookup tables for all used nomenclatures, and bibliographic information are bundled together. Data is organized into groups by sample type. The `publisher_postprocess_logs` global attribute records every curation step applied to the dataset, providing a permanent audit trail.

**CSV** (legacy format for database import)  
Flat files with one file per sample type, compatible with the MARIS master database ingestion pipeline.

## How data is curated

Data from providers arrives in many formats and uses provider-specific nomenclature. Before entering MARIS, each dataset is processed through the marisco curation pipeline:

1. **Nomenclature mapping** — nuclide names, species names, body parts, and sediment types are mapped to MARIS standard IDs using fuzzy-matched lookup tables, with manual review of uncertain matches.
2. **Unit standardisation** — seawater activities are converted to Bq m⁻³; biota activities carry a wet-weight or dry-weight flag; uncertainties are expressed as 1σ absolute values.
3. **Coordinate and date validation** — coordinates are converted to decimal degrees; dates are parsed to a common datetime; invalid or missing values are dropped.
4. **Detection limit encoding** — provider-specific flags are mapped to the four MARIS standard codes (`=`, `<`, `ND`, `DE`).
5. **Audit trail** — every curation step is logged and stored in the NetCDF4 file's `publisher_postprocess_logs` global attribute.

## Available datasets

| Dataset | Provider | Sample types | Coverage |
|---|---|---|---|
| [MARIS Legacy](../handlers/maris_legacy.ipynb) | MARIS Master Database | All types | Global, historical |
| [HELCOM](../handlers/helcom.ipynb) | HELCOM MORS | SEAWATER, BIOTA, SEDIMENT | Baltic Sea |
| [OSPAR](../handlers/ospar.ipynb) | ODIMS OSPAR | BIOTA | NE Atlantic |
| [TEPCO](../handlers/tepco.ipynb) | Fukushima monitoring (TEPCO / NRA) | SEAWATER, BIOTA, SEDIMENT | NW Pacific |
| [GEOTRACES](../handlers/geotraces.ipynb) | BODC GEOTRACES IDP2021 | SEAWATER | Global |

## Writing a new handler

A **handler** is a Jupyter notebook (`nbs/handlers/your_dataset.ipynb`) that encodes one data provider's raw files into a MARIS-standard NetCDF4 file. The notebook is the authoritative, literate-programming description of every curation decision made for that dataset; the Python module (`marisco/handlers/your_dataset.py`) is auto-generated from it via nbdev.

Use `helcom` as the reference — it covers all four sample types and every curation rule. OSPAR, GEOTRACES, and TEPCO are additional examples with different structural challenges.

→ Full technical reference: [Handler pattern (CLAUDE.md)](../handlers/CLAUDE.md)

### Prerequisites

- Raw source data from the provider and its format documentation
- A Zotero record key for the dataset (8-character alphanumeric; create a record in the [MARIS Zotero library](https://www.zotero.org/groups/2432820/maris/library) if one doesn't exist)
- MARIS lookup tables installed locally (`maris_init` downloads them to `~/.marisco/`)

### Handler anatomy

Every handler exposes a single CLI entry point — `encode(fname_out)` — built from six parts:

```python
# 1. Constants
src_dir = '...'           # path or URL to raw data
fname_out = '...'         # default output filename
zotero_key = 'XXXXXXXX'  # 8-char Zotero key

# 2. Data loader — returns dict keyed by sample type group
def load_data(...) -> Dict[str, pd.DataFrame]: ...

# 3. Lookup table factories — one per remapped enumerated field
lut_nuclides = lambda df: Remapper(...).generate_lookup_table(fixes=fixes_nuclide_names)

# 4. Callbacks — one class per transformation step
#    Callback docstring = audit trail entry in the NetCDF4 file
class RemapNuclideNameCB(PerGroupCB):
    "Remap provider nuclide names to MARIS standard IDs; unresolved names set to 0."
    ...

# 5. Global attributes builder
def get_attrs(tfm, zotero_key, kw) -> dict:
    return GlobAttrsFeeder(tfm.dfs, cbs=[BboxCB(), DepthRangeCB(), TimeRangeCB(),
        ZoteroCB(zotero_key, cfg=cfg()), ...])()

# 6. encode() — the CLI entry point
def encode(fname_out, **kwargs):
    dfs = load_data(src_dir)
    tfm = Transformer(dfs, cbs=[...])
    tfm()
    NetCDFEncoder(tfm.dfs, dest_fname=fname_out,
                  global_attrs=get_attrs(tfm, zotero_key, kw)).encode()
```

### Reconciling provider nomenclature — IMFA

Provider names for nuclides, species, body parts, and sediment types rarely match MARIS IDs. The **IMFA** pattern resolves this systematically for every enumerated field:

| Step | What happens |
|---|---|
| **Inspect** | List all distinct provider values for the field |
| **Match** | Run `Remapper` fuzzy-matching against the MARIS lookup table |
| **Fix** | Create a `fixes_*` dict for unmatched entries — this step requires domain knowledge; automated matching is never fully reliable |
| **Apply** | Freeze the mapping as a `lut_*` lambda; pass it to `RemapCB` (or a custom callback) in the `Transformer` |

For small, stable enumerations (e.g. a filtered/unfiltered flag), skip `Remapper` and write a plain dict directly.

### Key curation rules

| Rule | Detail |
|---|---|
| **Units — seawater** | Must be Bq m⁻³; multiply VALUE, UNC, and any detection-limit value by 1000 if the provider reports Bq L⁻¹ |
| **Units — biota** | Prefer wet weight (unit ID 5); also compute `PERCENTWT` when both wet and dry weights are available |
| **Uncertainty** | 1σ absolute, same unit as VALUE; convert from relative: `UNC = VALUE × UNC_pct / 100` |
| **Detection limit** | Map to MARIS IDs: `1` (= measured), `2` (< MDA), `3` (ND, no limit), `4` (derived/aggregated) |
| **Coordinates** | Decimal degrees; LON ∈ [−180, 180]; drop rows where LAT = LON = 0 (unknown position sentinel) |
| **Time** | Parse to `pd.Timestamp` first; encode to integer seconds since 1970-01-01 UTC via `EncodeTimeCB`; drop rows with unparseable dates |
| **Depth** | `SMP_DEPTH` in metres; use `−1` as sentinel for "not available" |
| **Unknown IDs** | Set to `0` (not `NaN`) for unresolved enumerated fields — never silently drop a row because a name could not be remapped |

### Verification checklist

Before considering a handler complete:

- [ ] All nuclide names resolve to a non-zero MARIS ID — check with `lut_nuclides()` and inspect `RemapCB` output
- [ ] `tfm.logs` contains one entry per callback (the audit trail is complete)
- [ ] Bounding box, depth range, and time range in `get_attrs()` look geographically and temporally plausible
- [ ] `decode(fname_out, verbose=True)` produces readable CSV files without unexpected column types
- [ ] Row counts before and after `SanitizeLonLatCB` and `EncodeTimeCB` are consistent with the expected data volume — use `CompareDfsAndTfmCB` to surface drops
- [ ] Negative activity values are flagged (radiometric artifact) rather than silently accepted

::: {.callout-note}
## Wishful thinking — what this section would ideally become

A few things that would make this guide significantly more useful, earmarked for future improvement:

- **A minimal worked example** — a "hello world" handler for a fictional two-column CSV provider, walking through every step in under 50 lines of code. Reading helcom to learn the pattern is valid but the full handler is 500+ lines with HELCOM-specific complexity.
- **A provider intake form** — a short checklist that a data provider fills in before the handler is written: What format is the data in? What are the nuclide names? Are there detection limit flags? What unit system? This would front-load the domain decisions before any code is written.
- **An interactive validation notebook** — a notebook that loads any `.nc` output file and runs the full checklist above programmatically, surfacing issues as structured warnings rather than requiring manual inspection.
- **Column mapping reference as a live table** — `NC_VARS` in `configs.ipynb` is authoritative but not easy to browse; a rendered table in this guide cross-referencing handler column names, NetCDF variable names, units, and lookup table would reduce context-switching.
:::

→ See [Handler pattern (CLAUDE.md)](../handlers/CLAUDE.md) for the complete field reference, column naming conventions, and data curation rules.

## Accessing MARIS data

- **Web interface** — [maris.iaea.org](https://maris.iaea.org): browse, search, filter, and download datasets
- **marisco** — this Python package generates MARIS-standard NetCDF4 files locally from the source datasets listed above; see the [getting started guide](../index.ipynb) for installation and usage